<a href="https://colab.research.google.com/github/rainforest01-coder/ESAA_files/blob/OB/week1%EC%88%98%EC%83%81%EC%9E%91_%EC%88%98%EB%A3%8C%EC%98%88%EC%B8%A1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 데이콘 x BDA 제 2회 학습자 수료 예측 AI 경진대회

https://dacon.io/competitions/official/236664/codeshare/13929?page=1&dtype=recent

-데이터 특징
> 혼합형 데이터, 결측치 매우 많음, F1스코어( 확률 예측 자체보다 0/1 결정 임계값이 성능 좌우), 작은 데이터 과적합 위험

1) 코드 흐름 요약

* 전처리: 결측치 처리, 타입 정리, 카테고리
* 인코딩, 멀티라벨 텍스트->Top-N이진화
모델링: Stratified 5 fold cross validation, Predict& Save Artifacts(모델 확률, 예측값 저장), Anchor+ Upper/Lower Flip 앙상블
> CatBoost -> RandomForest -> 1차 앙상블 -> MLP(Colab T4 GPU로 따로 실행) -> 2차 앙상블 -> Logistic Regression -> 3차 앙상블 -> LightGBM -> 4차 앙상블 -> ExtraTrees -> 5차 앙상블 -> SVM -> 6차 앙상블(최종 파일)
* 제출


2) 새롭게 알게 된 내용
* Anchor 기반 Upper/Lower Flip 앙상블: Anchor이 0인데 새 모델이 강하게 1로 예측하는 경우 rescue(1로 변경), 반대의 경우 Filter(0으로 변경)
> 점수 향상 시 새 파일이 다음 Anchor가 되는 방식
> 과적합 및 모델의 영향을 과하게 받는 문제 해결 가능

In [ ]:
# 주요 코드

# ============================================================
# 2. Ensemble Strategy: "LGBM Diagnostic Filter"
# ============================================================
UPPER_THR = 0.50
LOWER_THR = 0.255

change_0_to_1 = 0
change_1_to_0 = 0

for i in range(len(final_pred)):
    # Case 1: Rescue (Anchor 0 -> 1)
    if anchor_label[i] == 0 and lgbm_prob[i] > UPPER_THR:
        final_pred[i] = 1
        change_0_to_1 += 1

    # Case 2: Filter (Anchor 1 -> 0)
    elif anchor_label[i] == 1 and lgbm_prob[i] < LOWER_THR:
        final_pred[i] = 0
        change_1_to_0 += 1